# Memoria Viva — restauración en Google Colab

Este notebook prepara el proyecto, instala las dependencias sin reemplazar PyTorch/CUDA de Colab, carga los modelos en GPU y publica la interfaz Gradio. Elegí **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU** si está disponible. Colab asigna hardware según disponibilidad; el notebook se detiene con un mensaje si el runtime no tiene CUDA.

**Entrada del proyecto:** carpeta `memoria viva` en Drive, o subí el ZIP `memoria_viva_colab.zip` que acompaña este notebook. El disco de la VM de Colab es temporal.

El modelo genera píxeles nuevos al completar daños y estima los colores; no recupera evidencia histórica. Revisá el resultado antes de compartirlo.

In [ ]:
# Diagnóstico de hardware y entorno
import os, torch
print('Python:', __import__('sys').version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
else:
    raise RuntimeError('No hay GPU CUDA. En Colab seleccioná una T4 GPU y volvé a ejecutar este notebook.')

: 

In [ ]:
# Elegí una fuente: carpeta de Drive o subir el ZIP del proyecto.
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

candidatos = [Path('/content/drive/MyDrive/memoria viva'), Path('/content/drive/MyDrive/memoria_viva')]
PROYECTO = next((p for p in candidatos if (p / 'src/gradio_app.py').exists()), None)
if PROYECTO is None:
    print('No encontré la carpeta en Drive. Subí memoria_viva_colab.zip desde tu computadora.')
    from google.colab import files
    import zipfile
    subidos = files.upload()
    zips = [n for n in subidos if n.lower().endswith('.zip')]
    if not zips:
        raise FileNotFoundError('Subí el archivo memoria_viva_colab.zip')
    destino = Path('/content/memoria_viva')
    destino.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as zf:
        zf.extractall(destino)
    # Acepta ZIP con los archivos en la raíz o dentro de una carpeta contenedora.
    PROYECTO = destino if (destino / 'src/gradio_app.py').exists() else next(
        p for p in destino.iterdir() if p.is_dir() and (p / 'src/gradio_app.py').exists()
    )
print('Proyecto:', PROYECTO)
assert (PROYECTO / 'assets').exists(), 'No se encontró assets en el proyecto.'
os.environ['MEMORIA_VIVA_ROOT'] = str(PROYECTO)

In [ ]:
# Instalar librerías de la app. Se conserva el PyTorch con CUDA provisto por Colab.
%pip install -q -r "{PROYECTO / 'requirements-colab.txt'}"

In [ ]:
# Comprobar imágenes y cargar la app. Los pesos se descargan de Hugging Face
# en la primera inferencia; puede tardar varios minutos.
import sys
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))
from src.gradio_app import crear_app
demo = crear_app()
print('Interfaz preparada. La primera inferencia cargará difusión y RT-DETR en la GPU.')

In [ ]:
# Iniciar la app y obtener un enlace temporal de Gradio.
# Mantener esta celda ejecutándose durante la demo.
demo.queue().launch(share=True, debug=True)

## Flujo de uso
1. Cargá un ejemplo del proyecto o subí una foto propia.
2. Para reparar, pintá encima de rayones/manchas; el área pintada es la máscara de inpainting.
3. Activá reparación, colorización o ambas.
4. Revisá original, reparación, resultado, máscara, mapa de diferencias y análisis RT-DETR.
5. Descargá el resultado. Para demostrar límites, probá una máscara sobre un rostro o una zona sin información suficiente y comentá qué inventó o alteró el modelo.

Para detener la app, interrumpí la celda de lanzamiento. Los enlaces `share=True` son temporales mientras la sesión siga activa; no guardes fotos privadas en una demo pública.